<div align="center">

<a href="https://colab.research.google.com/github/utkukose/cyber-physical-systems-NB-lecture/blob/main/exams/midterm/MID3_water_tank.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm option 3: A water tank with a pump

**Cyber-Physical Systems (BIL-435 Siber Fiziksel Sistemler)**<br>Midterm project: A model and a controller for a cyber-physical system. Starter Colab notebook

[Project page](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/exams/midterm) | [Course repository](https://github.com/utkukose/cyber-physical-systems-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: The roof tank of a student dormitory, filled by a pump of 3 m³/h. Replace the values marked `# TODO` with the data of your own system. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The tank as an integrator

*Lecture: [Week 2, Differential equations: How physical quantities change](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-02/lecture.html#differential-equations)*

The tank on the roof has the cross-section $A$ in m². A pump fills it from a cistern with the flow $q_{in}$, and the
residents draw the demand $q_{out}$, both in m³/h. The level $h$ in m follows

$$A\,\frac{dh}{dt} = q_{in} - q_{out} .$$

The tank is an integrator: Any difference between the inflow and the demand accumulates in the level. The demand of the
building has a morning and an evening peak. The cell defines the demand of one day and checks the volumes. The pump
must deliver the daily demand in a few hours, and its flow must exceed the peak of the demand.

In [ ]:
# TODO: the tank, the pump and the demand of your building
A = 4.0                  # cross-section of the tank (m²)
H_OVERFLOW = 2.4         # level of the overflow pipe (m)
H_MIN = 0.5              # the lowest level allowed (m): a reserve for the residents
Q_PUMP = 3.0             # flow of the pump at full speed (m³/h)
DT = 1 / 60              # simulation step (h): one minute
HOURS = np.arange(0, 24, DT)

def demand(h):
    """Demand of 120 residents (m³/h): a base, a morning peak at 7:30 and an evening peak at 21:00."""
    morning = 1.6 * math.exp(-0.5 * ((h - 7.5) / 0.8) ** 2)
    evening = 1.2 * math.exp(-0.5 * ((h - 21.0) / 1.5) ** 2)
    return 0.15 + morning + evening

q = np.array([demand(h) for h in HOURS])
volume = q.sum() * DT
print(f"Daily demand {volume:.2f} m³, peak {q.max():.2f} m³/h at {HOURS[q.argmax()]:.2f} h")
print(f"Pump: {Q_PUMP} m³/h, {volume / Q_PUMP:.2f} hours of pumping per day")
print(f"Usable volume between H_MIN and the overflow: {A * (H_OVERFLOW - H_MIN):.2f} m³")
RESULTS["daily demand (m³)"] = round(volume, 2)
RESULTS["peak demand (m³/h)"] = round(q.max(), 2)
RESULTS["pumping hours per day"] = round(volume / Q_PUMP, 2)

## 2. Float switches as a hybrid automaton

*Lecture: [Week 3, Hybrid automata](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-03/lecture.html#hybrid-automata)*

The simplest controller uses two float switches. The pump starts when the level falls to $h_{on}$ and stops when the
level reaches $h_{off}$. The automaton has the modes *Pumping* and *Idle* with these guards, and in each mode the level
follows the model of Section 1. A third switch at the high-high level $h_{hh}$, below the overflow, raises an alarm.
The band between $h_{on}$ and $h_{off}$ sets the number of starts. The motor of a pump allows only a few starts per
hour, because each start heats its windings. The cell sweeps the band and counts the starts of one day and the largest
number of starts within any hour.

In [ ]:
# TODO: the float switches of your tank and the limit of the motor
H_ON, H_OFF = 1.2, 2.0           # the pump starts at H_ON and stops at H_OFF (m)
H_HH = 2.2                       # high-high alarm (m)
MAX_STARTS_PER_HOUR = 6          # limit of the motor

def run_floats(h_on, h_off):
    """One day with the float switches from a full tank at midnight; returns the log and the hours of the starts."""
    level, pumping, log, starts = h_off, False, [], []
    for t in HOURS:
        if not pumping and level <= h_on:            # guard from Idle to Pumping
            pumping = True
            starts.append(t)
        elif pumping and level >= h_off:             # guard from Pumping to Idle
            pumping = False
        q_in = Q_PUMP if pumping else 0.0
        log.append((t, level, q_in, demand(t)))
        level += DT * (q_in - demand(t)) / A
    return pd.DataFrame(log, columns=["hour", "level", "q_in", "q_out"]), starts

def max_per_hour(starts):
    """The largest number of starts within any window of one hour."""
    s = np.array(starts)
    return int(max(((s >= t) & (s < t + 1)).sum() for t in s)) if len(s) else 0

rows = []
for band in [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]:
    df, st = run_floats(H_OFF - band, H_OFF)
    rows.append({"band (m)": band, "starts per day": len(st), "most starts in an hour": max_per_hour(st),
                 "lowest level (m)": round(df.level.min(), 3), "highest level (m)": round(df.level.max(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

floats, starts = run_floats(H_ON, H_OFF)
print(f"h_on = {H_ON} m, h_off = {H_OFF} m: {len(starts)} starts, at most {max_per_hour(starts)} in an hour "
      f"(limit {MAX_STARTS_PER_HOUR}), lowest level {floats.level.min():.3f} m")
RESULTS["float switches: starts per day"] = len(starts)
RESULTS["float switches: most starts in an hour"] = max_per_hour(starts)
RESULTS["float switches: lowest level (m)"] = round(floats.level.min(), 3)

## 3. The level sensor: Sampling and averaging

*Lecture: [Week 4, Sampling: How often to look](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#sampling) and [Week 4, Quantisation and noise](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#quantisation)*

A variable-speed pump needs a continuous level reading. An ultrasonic sensor sends 4 to 20 mA for 0 to 3 m to an
analogue input of 12 bits, which gives the resolution $q = 3/4096$ m. The inflow makes waves on the surface with a
frequency $f_w$ of about 0.4 Hz. Sampled every $T_s$ seconds, a wave appears at the alias frequency

$$f_a = \left| f_w - k\,f_s \right|, \qquad f_s = 1/T_s ,$$

with the integer $k$ that makes $f_a$ smallest. A slow alias looks like a real change of the level and moves the pump.
The cell lists the alias for several sampling periods and simulates two minutes of readings. It then averages the fast
readings over a window of $N$ samples. The average reduces the noise by $\sqrt{N}$ and delays the reading by about
$(N - 1)\,T_s/2$. A window of a whole number of wave periods also cancels the waves.

In [ ]:
# TODO: the level sensor of your tank
RANGE_M = 3.0            # measuring range of the sensor (m)
BITS = 12                # resolution of the analogue input (bits)
SIGMA_M = 0.004          # noise of a single reading (m)
WAVE_HZ = 0.4            # frequency of the surface waves (Hz)
WAVE_M = 0.01            # amplitude of the waves (m)
TS_S = 0.25              # chosen sampling period (s)
N_AVG = 20               # length of the moving average (samples)

q_res = RANGE_M / 2 ** BITS
rng = np.random.default_rng(4)

def alias(f, fs):
    """The frequency at which a sampled sine wave of frequency f appears."""
    return abs(f - round(f / fs) * fs)

def readings(ts, seconds=120):
    """Readings of a level that rises with the pump at full flow, with waves, noise and quantisation."""
    t = np.arange(0, seconds, ts)
    level = 1.5 + Q_PUMP / A / 3600 * t + WAVE_M * np.sin(2 * np.pi * WAVE_HZ * t)
    return t, q_res * np.round((level + rng.normal(0, SIGMA_M, t.size)) / q_res)

print(f"Resolution q = {1000 * q_res:.2f} mm")
for ts in [2.4, 2.0, 1.0, TS_S]:
    print(f"T_s = {ts} s: f_s = {1 / ts:.3f} Hz, the waves appear at {alias(WAVE_HZ, 1 / ts):.4f} Hz")

t_fast, y_fast = readings(TS_S)
avg = np.convolve(y_fast, np.ones(N_AVG) / N_AVG, mode="valid")
t_avg = t_fast[N_AVG - 1:]
true = 1.5 + Q_PUMP / A / 3600 * t_avg
noise_raw = np.std(y_fast - (1.5 + Q_PUMP / A / 3600 * t_fast))
noise_avg = np.std(avg - true)
delay = (N_AVG - 1) * TS_S / 2
print(f"Spread of the raw readings {1000 * noise_raw:.2f} mm, of the average {1000 * noise_avg:.2f} mm, "
      f"window {N_AVG * TS_S:.1f} s = {N_AVG * TS_S * WAVE_HZ:.1f} wave periods, delay {delay:.2f} s")
RESULTS["resolution (mm)"] = round(1000 * q_res, 2)
RESULTS["alias at the chosen sampling (Hz)"] = round(alias(WAVE_HZ, 1 / TS_S), 4)
RESULTS["spread of the averaged reading (mm)"] = round(1000 * noise_avg, 2)
RESULTS["delay of the average (s)"] = delay

t_slow, y_slow = readings(2.4)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(t_fast, 1000 * (y_fast - 1.5), lw=0.6, color="#9AA5AE", label=f"T_s = {TS_S} s")
ax.plot(t_slow, 1000 * (y_slow - 1.5), "o-", ms=3, label="T_s = 2.4 s")
ax.plot(t_avg - delay, 1000 * (avg - 1.5), lw=2, label=f"average of {N_AVG}, shifted by its delay")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Level above 1.5 m (mm)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. A variable-speed pump with a proportional-integral controller

*Lecture: [Week 5, Integral action](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#pi-control) and [Week 5, Saturation and windup](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#windup)*

A frequency converter lets the pump run at any speed $u$ between 0 and 1, with the flow $q_{in} = u\,Q_{pump}$. A
proportional-integral (PI) controller keeps the level at the setpoint $r$:

$$u = K_p\,e + K_i \int e\,dt, \qquad e = r - h .$$

With the tank as an integrator, the closed loop has the characteristic equation
$A s^2 + K_p Q_{pump}\,s + K_i Q_{pump} = 0$. A damping ratio $\zeta$ and a natural frequency $\omega$ give

$$K_p = \frac{2\,\zeta\,\omega\,A}{Q_{pump}}, \qquad K_i = \frac{\omega^2 A}{Q_{pump}} .$$

Below a minimum speed $u_{min}$, a pump cannot lift water to the roof. The frequency converter then sleeps: It stops
the pump and wakes it when the level has fallen by the wake-up difference $\Delta h_w$ below the setpoint. Without the
sleep mode, the low demand of the night would start and stop the pump every few minutes. The anti-windup stops the
integration while the speed is saturated in the direction of the error or the pump sleeps. The controller reads the
averaged level of Section 3 once a minute. The cell compares the PI controller with the float switches of Section 2.

In [ ]:
# TODO: the design of your PI controller
SETPOINT = 1.6           # level setpoint (m)
ZETA = 1.0               # damping ratio of the closed loop
T_CL = 0.25              # time constant of the closed loop, 1 / omega (h)
U_MIN = 0.2              # lowest speed at which the pump delivers water
WAKE = 0.15              # wake-up difference below the setpoint (m)

omega = 1 / T_CL
KP = 2 * ZETA * omega * A / Q_PUMP       # proportional gain (1/m)
KI = omega ** 2 * A / Q_PUMP             # integral gain (1/(m h))
print(f"Kp = {KP:.2f} 1/m, Ki = {KI:.2f} 1/(m h)")

def run_pi(kp, ki, wake=WAKE, seed=5):
    """One day with the variable-speed pump from the setpoint at midnight; returns the log and the hours of the starts."""
    rng = np.random.default_rng(seed)
    level, I, u, asleep, log, starts = SETPOINT, 0.0, 0.0, True, [], []
    for t in HOURS:
        y = level + rng.normal(0, noise_avg)         # the averaged reading of Section 3
        e = SETPOINT - y
        if asleep and e >= wake:                     # wake up
            asleep = False
            starts.append(t)
        v = kp * e + ki * I
        if not asleep and v < U_MIN:                 # too slow to lift the water: sleep
            asleep = True
        u = 0.0 if asleep else min(v, 1.0)
        saturated = asleep or (v > 1.0 and e > 0)
        if not saturated:
            I += e * DT
        log.append((t, level, u * Q_PUMP, demand(t)))
        level += DT * (u * Q_PUMP - demand(t)) / A
    return pd.DataFrame(log, columns=["hour", "level", "q_in", "q_out"]), starts

pi, pi_starts = run_pi(KP, KI)
rows = []
for label, df, st in [(f"float switches {H_ON} to {H_OFF} m", floats, starts), ("PI, variable speed", pi, pi_starts)]:
    rows.append({"controller": label, "starts per day": len(st), "most starts in an hour": max_per_hour(st),
                 "lowest level (m)": round(df.level.min(), 3), "highest level (m)": round(df.level.max(), 3),
                 "spread of the level (m)": round(df.level.std(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
RESULTS["PI: starts per day"] = len(pi_starts)
RESULTS["PI: most starts in an hour"] = max_per_hour(pi_starts)
RESULTS["PI: lowest level (m)"] = round(pi.level.min(), 3)
RESULTS["PI: spread of the level (m)"] = round(pi.level.std(), 3)

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(floats.hour, floats.level, label="float switches")
ax[0].plot(pi.hour, pi.level, label="PI, variable speed")
ax[0].axhline(H_MIN, color="#B23B3B", ls="--", lw=1, label="lowest allowed")
ax[1].plot(floats.hour, floats.q_in, lw=0.8, label="pump, float switches")
ax[1].plot(pi.hour, pi.q_in, lw=0.8, label="pump, PI")
ax[1].plot(pi.hour, pi.q_out, "k--", lw=1, label="demand")
ax[0].set_ylabel("Level (m)")
ax[1].set_ylabel("Flow (m³/h)")
ax[1].set_xlabel("Hour of the day")
for a in ax:
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. The scan cycle and the time to the overflow

*Lecture: [Week 7, The scan cycle of a programmable logic controller](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-07/lecture.html#plc-scan)*

A programmable logic controller (PLC) runs the controller in a scan cycle: It reads the inputs, executes the program and
writes the outputs. A change at an input waits for the next read and reaches the output at the end of that scan, so the
reaction time $t_r$ lies in

$$T_{scan} + t_{in} + t_{out} \le t_r < 2\,T_{scan} + t_{in} + t_{out},$$

where $t_{in}$ is the delay of the input filter and $t_{out}$ that of the output relay. After the output, the contactor
opens and the pump runs down. During the whole stop time, the level keeps rising at up to $Q_{pump}/A$. A reading over a
network adds its poll cycle $T_{poll}$. The cell checks the scan times logged by the PLC against its watchdog. It
computes the stop time and the rise of the level for a hardwired switch and for a reading over the network. Finally, it
computes the time from the high-high alarm to the overflow when the contactor fails to open. Within this time, the alarm
must reach a person who can stop the pump.

In [ ]:
# TODO: the PLC, the actuators and the alarm path of your system
SCAN_LOG_MS = [4.1, 4.3, 4.2, 4.8, 4.2, 4.1, 5.6, 4.2, 4.3, 4.1, 4.2, 4.4, 4.2, 6.3, 4.1, 4.2, 4.3, 4.2, 4.5, 4.2]
WATCHDOG_MS = 50         # the PLC stops if a scan takes longer
T_IN_MS, T_OUT_MS = 3, 6       # input filter and output relay (ms)
T_CONTACTOR_MS = 40      # opening time of the contactor (ms)
T_RUNDOWN_S = 3.0        # run-down of the pump until the flow stops (s)
T_POLL_S = 2.0           # poll cycle of a level reading over the network (s)
T_ALARM_MIN = 10.0       # time for the alarm to reach a person who can stop the pump (min)

t_scan = max(SCAN_LOG_MS)
t_react = 2 * t_scan + T_IN_MS + T_OUT_MS                    # worst-case reaction of the PLC (ms)
rise_rate = Q_PUMP / A / 3600                                 # rise of the level with no demand (m/s)
print(f"Scan time: mean {np.mean(SCAN_LOG_MS):.2f} ms, longest {t_scan} ms, watchdog {WATCHDOG_MS} ms")
print(f"Worst-case reaction of the PLC: {t_react:.1f} ms")
rows = []
for label, extra in [("hardwired switch", 0.0), ("reading over the network", T_POLL_S)]:
    t_stop = extra + (t_react + T_CONTACTOR_MS) / 1000 + T_RUNDOWN_S
    rows.append({"path of the reading": label, "stop time (s)": round(t_stop, 3),
                 "rise of the level (mm)": round(1000 * rise_rate * t_stop, 2)})
print(pd.DataFrame(rows).to_string(index=False))
t_overflow = (H_OVERFLOW - H_HH) / rise_rate / 60             # minutes from the alarm to the overflow
print(f"From the high-high alarm to the overflow: {t_overflow:.1f} min; the alarm path needs {T_ALARM_MIN} min")
RESULTS["worst-case reaction of the PLC (ms)"] = round(t_react, 1)
RESULTS["stop time, hardwired (s)"] = rows[0]["stop time (s)"]
RESULTS["rise of the level, hardwired (mm)"] = rows[0]["rise of the level (mm)"]
RESULTS["time from the alarm to the overflow (min)"] = round(t_overflow, 1)
RESULTS["alarm path fast enough"] = bool(T_ALARM_MIN < t_overflow)

The PLC reacts in milliseconds, while the tank fills in minutes. Discuss which part of the chain sets the safety margin of your system, and what happens when the high-high switch itself fails.

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>